<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade3_descontos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 3 — Functions de desconto
Execute a única célula de código. Ela instala o MariaDB, cria o arquivo SQL e mostra os resultados.

In [1]:
import os, shutil, subprocess, time
from pathlib import Path

# Instala o MariaDB caso ele não exista na sessão atual
if shutil.which("mariadb") is None:
    print("Instalando MariaDB...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "mariadb-server", "mariadb-client"],
        check=True,
        stdout=subprocess.DEVNULL
    )

# Inicia o servidor
os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(["chown", "mysql:mysql", "/run/mysqld"], check=False)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(20):
    teste = subprocess.run(
        ["mariadb-admin", "-u", "root", "ping", "--silent"],
        text=True,
        capture_output=True
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Não foi possível iniciar o MariaDB.")

print("MariaDB iniciado com sucesso.")

sql = r"""-- ATIVIDADE 3 – FUNCTIONS DE DESCONTO
DROP DATABASE IF EXISTS atividade_descontos;
CREATE DATABASE atividade_descontos CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE atividade_descontos;

CREATE TABLE produto (
    id_produto INT AUTO_INCREMENT PRIMARY KEY,
    preco DECIMAL(10,2) NOT NULL,
    CONSTRAINT chk_produto_preco CHECK (preco > 0)
) ENGINE=InnoDB;

INSERT INTO produto (preco) VALUES
(80.00), (100.00), (150.00), (250.00);

DROP FUNCTION IF EXISTS calcular_desconto;
DROP FUNCTION IF EXISTS preco_final;

DELIMITER $$

CREATE FUNCTION calcular_desconto(p_preco DECIMAL(10,2))
RETURNS DECIMAL(10,2)
DETERMINISTIC
BEGIN
    IF p_preco IS NULL OR p_preco <= 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'O preço deve ser um valor positivo.';
    END IF;

    IF p_preco > 100 THEN
        RETURN ROUND(p_preco * 0.10, 2);
    ELSE
        RETURN ROUND(p_preco * 0.05, 2);
    END IF;
END$$

CREATE FUNCTION preco_final(p_preco DECIMAL(10,2))
RETURNS DECIMAL(10,2)
DETERMINISTIC
BEGIN
    IF p_preco IS NULL OR p_preco <= 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'O preço deve ser um valor positivo.';
    END IF;

    RETURN ROUND(p_preco - calcular_desconto(p_preco), 2);
END$$

DELIMITER ;
"""

arquivo = Path("/content/atividade3_descontos.sql")
arquivo.write_text(sql, encoding="utf-8")
print("Arquivo criado:", arquivo)

with arquivo.open("r", encoding="utf-8") as entrada:
    execucao = subprocess.run(
        ["mariadb", "-u", "root", "--default-character-set=utf8mb4"],
        stdin=entrada,
        text=True,
        capture_output=True
    )

if execucao.returncode != 0:
    print(execucao.stderr)
    raise RuntimeError("Erro ao executar o script SQL.")

consultas = r"""
USE atividade_descontos;

SELECT
    id_produto,
    preco AS preco_original,
    CASE WHEN preco > 100 THEN '10%' ELSE '5%' END AS percentual_desconto,
    calcular_desconto(preco) AS valor_desconto,
    preco_final(preco) AS preco_final
FROM produto
ORDER BY id_produto;

SHOW FUNCTION STATUS
WHERE Db = 'atividade_descontos';
"""

resultado = subprocess.run(
    ["mariadb", "-u", "root", "--table", "--default-character-set=utf8mb4", "-e", consultas],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("Erro ao consultar os resultados.")

print("Atividade 3 executada com sucesso.")


Instalando MariaDB...
MariaDB iniciado com sucesso.
Arquivo criado: /content/atividade3_descontos.sql
+------------+----------------+---------------------+----------------+-------------+
| id_produto | preco_original | percentual_desconto | valor_desconto | preco_final |
+------------+----------------+---------------------+----------------+-------------+
|          1 |          80.00 | 5%                  |           4.00 |       76.00 |
|          2 |         100.00 | 5%                  |           5.00 |       95.00 |
|          3 |         150.00 | 10%                 |          15.00 |      135.00 |
|          4 |         250.00 | 10%                 |          25.00 |      225.00 |
+------------+----------------+---------------------+----------------+-------------+
+---------------------+-------------------+----------+----------------+---------------------+---------------------+---------------+---------+----------------------+----------------------+--------------------+
| Db     